# 使用Amazon EC2实例
:label:`sec_aws`

本节将展示如何在原始Linux机器上安装所有库。回想一下， :numref:`sec_sagemaker`讨论了如何使用Amazon SageMaker，而在云上自己构建实例的成本更低。本演示包括三个步骤。

1. 从AWS EC2请求GPU Linux实例。
1. 安装CUDA（或使用预装CUDA的Amazon机器映像）。
1. 安装深度学习框架和其他库以运行本书的代码。

此过程也适用于其他实例（和其他云），尽管需要一些细微的修改。在继续操作之前，你需要创建一个AWS帐户，有关更多详细信息，请参阅 :numref:`sec_sagemaker`。

## 创建和运行EC2实例

登录到你的aws账户后，单击"EC2"（在 :numref:`fig_aws`中用红色方框标记）进入EC2面板。

![打开EC2控制台](../img/aws.png)
:width:`400px`
:label:`fig_aws`

:numref:`fig_ec2`显示EC2面板，敏感帐户信息变为灰色。

![EC2面板](../img/ec2.png)
:width:`700px`
:label:`fig_ec2`

### 预置位置
选择附近的数据中心以降低延迟，例如"Oregon"（俄勒冈）( :numref:`fig_ec2`右上角的红色方框）。如果你位于中国，你可以选择附近的亚太地区，例如首尔或东京。请注意，某些数据中心可能没有GPU实例。

### 增加限制

在选择实例之前，请点击 :numref:`fig_ec2`所示左侧栏中的"Limits"（限制）标签查看是否有数量限制。 :numref:`fig_limits`显示了此类限制的一个例子。账号目前无法按地域打开p2.xlarge实例。如果你需要打开一个或多个实例，请点击"Request limit increase"（请求增加限制）链接，申请更高的实例配额。一般来说，需要一个工作日的时间来处理申请。

![实例数量限制](../img/limits.png)
:width:`700px`
:label:`fig_limits`

### 启动实例

接下来，单击 :numref:`fig_ec2`中红框标记的"Launch Instance"（启动实例）按钮，启动你的实例。

我们首先选择一个合适的Amazon机器映像（Amazon Machine Image，AMI）。在搜索框中输入"ubuntu"（ :numref:`fig_ubuntu`中的红色框标记）。

![选择一个AMI](../img/ubuntu-new.png)
:width:`700px`
:label:`fig_ubuntu`

### 推薦作業系統（2024更新）

建議使用以下作業系統：

* **Ubuntu 22.04 LTS**：長期支援版本，穩定且廣泛使用
* **Ubuntu 24.04 LTS**：最新 LTS 版本，支持最新硬體
* **Deep Learning AMI (Ubuntu)**：AWS 預配置的深度學習環境
* **Deep Learning AMI (Amazon Linux 2)**：針對 AWS 優化的選項

EC2提供了许多不同的实例配置可供选择。对初学者来说，这有时会让人感到困惑。 :numref:`tab_ec2`列出了不同合适的计算机。

:不同的EC2实例类型（2024更新）

| 實例類型 | GPU | 顯存 | 用途 | 價格等級 | 發布年份 |
|---------|-----|------|------|---------|---------|
| g4dn.xlarge | T4 | 16GB | 推理優化 | $ | 2019 |
| g5.xlarge | A10G | 24GB | 訓練/推理 | $$ | 2021 |
| g5.2xlarge | A10G | 24GB | 訓練（更多RAM） | $$$ | 2021 |
| p3.2xlarge | V100 | 16GB | 高性能訓練 | $$$ | 2017 |
| p4d.24xlarge | A100 | 40GB × 8 | 大規模訓練 | $$$$ | 2020 |
| p5.48xlarge | H100 | 80GB × 8 | 超大規模訓練 | $$$$$ | 2023 |

:label:`tab_ec2`

**舊版實例（不推薦用於新項目）**：
- g2 (Grid K520) - 已過時
- p2 (Kepler K80) - 舊 GPU，但 Spot 實例價格便宜
- g3 (Maxwell M60) - 已被 g4/g5 取代

所有这些服务器都有多种类型，显示了使用的GPU数量。例如，g5.xlarge有1个GPU，而p4d.24xlarge有8个A100 GPU和1152GB系统内存。有关更多详细信息，请参阅[Amazon EC2 文档](https://aws.amazon.com/ec2/instance-types/)。

### 實例選擇建議（2024）

- **入門學習**：g4dn.xlarge (T4, 16GB) - 性價比高，適合學習和小型項目
- **中型訓練**：g5.xlarge/2xlarge (A10G, 24GB) - 平衡性能和成本，適合大多數深度學習任務
- **大型模型**：p4d.24xlarge (8×A100, 40GB) - 專業級訓練，適合大規模模型
- **前沿研究**：p5.48xlarge (8×H100, 80GB) - 最新最強，適合超大模型和尖端研究

![选择一个实例](../img/p2x.png)
:width:`700px`
:label:`fig_p2x`

注意，你应该使用支持GPU的实例以及合适的驱动程序和支持GPU的深度学习框架。否则，你将感受不到使用GPU的任何好处。

到目前为止，我们已经完成了启动EC2实例的七个步骤中的前两个步骤，如 :numref:`fig_disk`顶部所示。在本例中，我们保留"3. Configure Instance"（3. 配置实例）、"5. Add Tags"（5. 添加标签）和"6. Configure Security Group"（6. 配置安全组）步骤的默认配置。点击"4.添加存储"并将默认硬盘大小增加到至少 100GB（ :numref:`fig_disk`中的红色框标记）。請注意，現代深度學習框架和 CUDA 工具包可能需要 10-20GB 空間，而大型數據集和模型檢查點需要額外的存儲空間。

![修改硬盘大小](../img/disk.png)
:width:`700px`
:label:`fig_disk`

最后，进入"7. Review"（7. 查看），点击"Launch"（启动），即可启动配置好的实例。系统现在将提示你选择用于访问实例的密钥对。如果你没有密钥对，请在 :numref:`fig_keypair`的第一个下拉菜单中选择"Create a new key pair"（新建密钥对），即可生成密钥对。之后，你可以在此菜单中选择"Choose an existing key pair"（选择现有密钥对），然后选择之前生成的密钥对。单击"Launch Instances"（启动实例）即可启动创建的实例。

![选择一个密钥对](../img/keypair.png)
:width:`500px`
:label:`fig_keypair`

如果生成了新密钥对，请确保下载密钥对并将其存储在安全位置。这是你通过SSH连接到服务器的唯一方式。单击 :numref:`fig_launching`中显示的实例ID可查看该实例的状态。

![单击实例ID](../img/launching.png)
:width:`700px`
:label:`fig_launching`

### 连接到实例

如 :numref:`fig_connect`所示，实例状态变为绿色后，右键单击实例，选择`Connect`（连接）查看实例访问方式。

![查看实例访问方法](../img/connect.png)
:width:`700px`
:label:`fig_connect`

如果这是一个新密钥，它必须是不可公开查看的，SSH才能工作。转到存储`D2L_key.pem`的文件夹，并执行以下命令以使密钥不可公开查看：

```bash
chmod 400 D2L_key.pem
```

![查看实例访问和启动方法](../img/chmod.png)
:width:`400px`
:label:`fig_chmod`

现在，复制 :numref:`fig_chmod`下方红色框中的ssh命令并粘贴到命令行：

```bash
ssh -i "D2L_key.pem" ubuntu@ec2-xx-xxx-xxx-xxx.y.compute.amazonaws.com
```

当命令行提示"Are you sure you want to continue connecting (yes/no)"（"你确定要继续连接吗？（是/否）"）时，输入"yes"并按回车键登录实例。

你的服务器现在已就绪。

## 安装CUDA

在安装CUDA之前，请确保使用最新的驱动程序更新实例。

```bash
sudo apt-get update && sudo apt-get install -y build-essential git libgfortran5
```

### CUDA 版本建議（2024更新）

**當前穩定版本**：
- **CUDA 12.1+**：推薦用於 PyTorch 2.5+ 和 TensorFlow 2.15+
- **CUDA 11.8**：如果需要向後兼容性

我們在這裡展示如何安裝 CUDA 12.1。訪問NVIDIA的[官方存储库](https://developer.nvidia.com/cuda-toolkit-archive) 以找到下载链接。

```bash
# CUDA 12.1 安裝示例（Ubuntu 22.04）
# 具體指令請參考 NVIDIA 官方文檔
wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.0-1_all.deb
sudo dpkg -i cuda-keyring_1.0-1_all.deb
sudo apt-get update
sudo apt-get -y install cuda-12-1
```

### 使用 Deep Learning AMI（推薦）

**更簡單的方式**：使用 AWS Deep Learning AMI，已預裝：
- CUDA（多個版本）
- cuDNN
- PyTorch、TensorFlow、MXNet
- Jupyter Notebook
- 其他常用深度學習庫

選擇 AMI 時搜索 "Deep Learning AMI GPU PyTorch" 或 "Deep Learning AMI GPU TensorFlow"。

安装程序后，运行以下命令查看GPU：

```bash
nvidia-smi
```

最后，将CUDA添加到库路径以帮助其他库找到它。

```bash
echo "export LD_LIBRARY_PATH=\${LD_LIBRARY_PATH}:/usr/local/cuda/lib64" >> ~/.bashrc
source ~/.bashrc
```

## 安装库以运行代码

### 使用 Deep Learning AMI（推薦）

如果使用 Deep Learning AMI，大多數庫已預裝。只需激活相應的 conda 環境：

```bash
# 激活 PyTorch 環境
conda activate pytorch

# 或激活 TensorFlow 環境
conda activate tensorflow
```

### 手動安裝

要运行本书的代码，只需在EC2实例上为linux用户执行 :ref:`chap_installation`中的步骤，并使用以下提示在远程linux服务器上工作。

* 要在Miniconda安装页面下载bash脚本，请右击下载链接并选择"copy Link address"，然后执行`wget [copied link address]`。
* 运行`~/miniconda3/bin/conda init`, 你可能需要执行`source ~/.bashrc`，而不是关闭并重新打开当前shell。

```bash
# 安裝 PyTorch（CUDA 12.1）
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 驗證安裝
python -c "import torch; print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')"
```

## 远程运行Jupyter笔记本

要远程运行Jupyter笔记本，你需要使用SSH端口转发。毕竟，云中的服务器没有显示器或键盘。为此，请从你的台式机（或笔记本电脑）登录到你的服务器，如下所示：

```
# 此命令必须在本地命令行中运行
ssh -i "/path/to/key.pem" ubuntu@ec2-xx-xxx-xxx-xxx.y.compute.amazonaws.com -L 8889:localhost:8888
```

接下来，转到EC2实例上本书下载的代码所在的位置，然后运行：

```
conda activate d2l
jupyter notebook
```

:numref:`fig_jupyter`显示了运行Jupyter笔记本后可能的输出。最后一行是端口8888的URL。

![运行Jupyter Notebook后的输出（最后一行是端口8888的URL）](../img/jupyter.png)
:width:`700px`
:label:`fig_jupyter`

由于你使用端口转发到端口8889，请复制 :numref:`fig_jupyter`红色框中的最后一行，将URL中的"8888"替换为"8889"，然后在本地浏览器中打开它。

## 其他雲端 GPU 平台

除了 AWS，還有許多其他雲端 GPU 選項可供選擇：

### 免費/低成本選項

1. **Google Colab**（詳見 :numref:`sec_colab`）
   - 免費 T4 GPU
   - 適合學習和小型項目
   - 網址：https://colab.research.google.com

2. **Kaggle Notebooks**
   - 免費 P100/T4 GPU
   - 每週 30 小時 GPU 時間
   - 整合 Kaggle 數據集
   - 網址：https://www.kaggle.com/code

3. **Lightning AI**
   - 提供免費額度
   - 專為深度學習優化
   - 網址：https://lightning.ai

### 付費雲端 GPU

1. **Lambda Labs**
   - 性價比高的 GPU 雲端服務
   - 專注於深度學習
   - 按需和預留實例
   - 網址：https://lambdalabs.com

2. **RunPod**
   - 按需計費，彈性定價
   - 支持多種 GPU
   - 社區雲端選項更便宜
   - 網址：https://www.runpod.io

3. **Paperspace Gradient**
   - 完整的 ML 開發平台
   - Jupyter Notebook 整合
   - 免費和付費層級
   - 網址：https://www.paperspace.com/gradient

4. **Microsoft Azure ML**
   - 企業級 ML 平台
   - NC、ND、NV 系列 GPU 實例
   - 整合 Azure 生態系統
   - 網址：https://azure.microsoft.com/services/machine-learning

5. **Google Cloud Platform (GCP)**
   - N1、A2 GPU 實例
   - TPU 選項
   - 整合 Google 服務
   - 網址：https://cloud.google.com

### 成本優化建議

1. **使用 Spot/Preemptible 實例**：節省高達 70-90% 成本
2. **選擇合適的區域**：不同區域價格可能差異較大
3. **監控使用情況**：設置自動關機以避免浪費
4. **使用預留實例**：長期使用可獲得折扣
5. **比較多個平台**：根據需求選擇最具成本效益的選項

## 关闭未使用的实例

由于云服务是按使用时间计费的，你应该关闭不使用的实例。请注意，还有其他选择：

* "Stopping"（停止）实例意味着你可以重新启动它。这类似于关闭常规服务器的电源。但是，停止的实例仍将按保留的硬盘空间收取少量费用；
* "Terminating"（终止）实例将删除与其关联的所有数据。这包括磁盘，因此你不能再次启动它。只有在你知道将来不需要它的情况下才这样做。

如果你想要将该实例用作更多实例的模板，请右击 :numref:`fig_connect`中的例子，然后选择"Image"$\rightarrow$"Create"以创建该实例的镜像。完成后，选择"实例状态"$\rightarrow$"终止"以终止实例。下次要使用此实例时，可以按照本节中的步骤基于保存的镜像创建实例。唯一的区别是，在 :numref:`fig_ubuntu`所示的"1.选择AMI"中，你必须使用左侧的"我的AMI"选项来选择你保存的镜像。创建的实例将保留镜像硬盘上存储的信息。例如，你不必重新安装CUDA和其他运行时环境。

## 小结

* AWS EC2 提供多種 GPU 實例，從入門級的 T4 到高端的 H100，滿足不同需求
* 推薦使用 Ubuntu 22.04/24.04 LTS 或 Deep Learning AMI 以獲得最佳兼容性
* CUDA 12.1+ 是當前推薦版本，適用於最新的深度學習框架
* Deep Learning AMI 提供預配置環境，可大幅簡化設置過程
* 除了 AWS，還有多個雲端平台可選，包括免費選項（Colab、Kaggle）和付費服務（Lambda Labs、RunPod 等）
* 使用 Spot 實例和監控工具可以顯著降低雲端運算成本
* 我們可以使用端口转发在远程服务器上运行Jupyter笔记本

## 练习

1. 比較 g4dn.xlarge (T4) 和 g5.xlarge (A10G) 實例在相同任務上的性能和成本差異
2. 了解如何启动[Spot实例](https://aws.amazon.com/ec2/spot/)以降低成本，並設置自動重啟機制
3. 嘗試使用 Deep Learning AMI，比較與手動安裝的便利性差異
4. 在 AWS、Lambda Labs 和 RunPod 之間比較相同 GPU 配置的價格
5. 設置自動化腳本來監控實例使用情況並在空閒時自動關閉

[Discussions](https://discuss.d2l.ai/t/5733)